# 앙상블

## 가. 앙상블 학습 (Ensemble Learning) 과 투표/평균 기반 예측
### 1.  **개념**: 여러 개의 단일 결정 트리 모델을 연결하여 더 강력한 예측 성능을 만드는 기법
### 2. **주요 종류**: 
#### 가) **Bagging** (Bootstrap Aggregating)
#### 나) **Boosting** (부스팅)
#### 다) **Stacking** (스태킹)

### 3. 분류 문제: 투표 기반 분류 (Voting)
### 가) **직접 투표 (Hard Voting)**: 각 개별 분류 모델이 예측한 최종 클래스 레이블들의 다수결(Majority Vote)로 최종 클래스 결정
### 나) **간접 / 소프트 투표 (Soft Voting)**: 각 개별 모델이 출력하는 클래스별 예측 확률의 평균을 산출하여, 가장 높은 확률의 클래스를 선택 (일반적으로 Hard Voting보다 성능 우수)

### 4. 회귀 문제: 평균 기반 회귀 (Averaging)
* 연속형 개별 예측값이 출력되므로 투표 대신 각 개별 회귀 모델들이 예측한 **연속값들의 평균**을 계산하여 최종 예측값으로 결정

## 나. 샘플링 및 특성 추출 방식 (Bagging, Pasting, Random Patches)
### 1. **베깅 (Bagging)**: 학습용 데이터에서 **복원 추출 방식(중복 허용)**으로 서브 데이터를 생성하여 개별 훈련 후 집계
### 2. **페이스팅 (Pasting)**: 학습용 데이터에서 **비복원 추출 방식(중복 없음)**으로 서브 데이터 생성
### 3. **랜덤 패치 (Random Patches)**: '행(샘플)'과 '특성(열)'을 모두 무작위 추출 (`bootstrap=True & max_features < 1.0`)
### 4. **랜덤 서브스페이스 (Random Subspaces)**: 샘플은 전체를 사용하고, 특성만 무작위 추출 (`bootstrap=False & max_features < 1.0`)

## 다. 랜덤 포레스트 (Random Forest)
### 1. **개념**: 결정 트리(Decision Tree)를 기본 모델(Base Estimator)로 사용하는 랜덤 패치 기반의 대표적 베깅 앙상블
### 2. **트리의 무작위성 (Feature Subspaces at Nodes)**: 노드를 분할할 때 전체 특성을 검토하지 않고, 무작위로 선택된 특성의 부분집합($\sqrt{n\_features}$) 중에서 최선의 분할 특성을 선택

In [1]:
# 랜덤포레스트 분류

In [2]:
# 필요한 라이브러리 임포트
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
# 1. 데이터 불러오기 (당뇨병 건강 지표 데이터)
df_diabetes = pd.read_csv('./실습데이터/diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

In [4]:
# 2. 독립변수(X)와 종속변수(y) 분리
X_clf = df_diabetes.drop(columns=['Diabetes_binary'])
y_clf = df_diabetes['Diabetes_binary']

In [5]:
# 3. 학습용 데이터와 평가용 데이터 분할 (8:2 비율)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=0
)

In [6]:
# 4. 랜덤 포레스트 분류 모델 생성 및 학습
rf_clf = RandomForestClassifier(random_state=0)
rf_clf.fit(X_train_c, y_train_c)
print()

In [7]:
# 5. 평가용 데이터를 이용한 예측 및 정확도(Accuracy) 측정
pred_test_rf_clf = rf_clf.predict(X_test_c)
acc_rf = accuracy_score(y_test_c, pred_test_rf_clf)

print(f"Random Forest Classifier Accuracy: {acc_rf:.4f} ({acc_rf * 100:.2f}%)")

Random Forest Classifier Accuracy: 0.7375 (73.75%)


In [8]:
# 랜덤포레스트 회귀

In [9]:
# 필요한 라이브러리 임포트
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse

In [10]:
# 1. 데이터 불러오기 (보스턴 집값 데이터)
df_house = pd.read_csv('./실습데이터/house_price.csv')

In [11]:
# 2. 독립변수(X)와 종속변수(y) 분리
X_reg = df_house.drop(columns=['medv'])
y_reg = df_house['medv']

In [12]:
# 3. 학습용 데이터와 평가용 데이터 분할 (8:2 비율)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=0
)

In [13]:
# 4. 랜덤 포레스트 회귀 모델 생성 및 학습
rf_reg = RandomForestRegressor(random_state=0)
rf_reg.fit(X_train_r, y_train_r)
print()

In [14]:
# 5. 평가용 데이터를 이용한 예측 및 성능 평가 (RMSE)
pred_test_rf_reg = rf_reg.predict(X_test_r)
rmse_rf = rmse(y_test_r, pred_test_rf_reg)

print(f"Random Forest Regressor RMSE: {rmse_rf:.4f}")

Random Forest Regressor RMSE: 4.2979


## 라. 부스팅 (Boosting)과 XGBoost (Extreme Gradient Boosting)
### 1. **부스팅(Boosting)의 핵심 메커니즘**
* **순차적 학습 (Sequential Learning)**: 이전 모델의 오류(Residual)를 다음 모델이 보완하는 체인 형태의 앙상블 알고리즘

### 2. **'조각상 다듬기' 비유**
* **초기값**: 대략적인 큰 덩어리를 잡음 (첫 번째 트리 예측: 평균값으로 오차 계산)
* **1단계**: 튀어나온 큰 오차 부분을 깎아냄 (두 번째 트리 예측 $\times$ 학습률)
* **2단계**: 아직 덜 깎이거나 잘못 깎인 미세한 오차를 보고 조금 더 깎아냄 (세 번째 트리 예측 $\times$ 학습률)
* 이를 수백 번 반복하여 점점 정밀한 조각(완성된 모델)을 만들어 냄

In [15]:
# XGBoost 분류

In [16]:
# 필요한 라이브러리 임포트
from xgboost import XGBClassifier

In [17]:
# 1. XGBoost 분류 모델 생성 및 학습
xgb_clf1 = XGBClassifier(random_state=0, n_estimators=29)
xgb_clf1.fit(X_train_c, y_train_c)
print()

In [18]:
# 2. 평가용 데이터를 이용한 예측 및 정확도(Accuracy) 측정
pred_test_xgb_clf = xgb_clf1.predict(X_test_c)
acc_xgb1 = accuracy_score(y_test_c, pred_test_xgb_clf)

print(f"XGBoost Classifier Accuracy: {acc_xgb1:.4f} ({acc_xgb1 * 100:.2f}%)")

XGBoost Classifier Accuracy: 0.7544 (75.44%)


In [19]:
# XGBoost 회귀

In [20]:
# 필요한 라이브러리 임포트
from xgboost import XGBRegressor

In [21]:
# 1. XGBoost 회귀 모델 생성 및 학습
xgb_reg1 = XGBRegressor(random_state=0)
xgb_reg1.fit(X_train_r, y_train_r)
print()

In [22]:
# 2. 평가용 데이터를 이용한 예측 및 성능 평가 (RMSE)
pred_test_xgb_reg1 = xgb_reg1.predict(X_test_r)
rmse_xgb = rmse(y_test_r, pred_test_xgb_reg1)

print(f"XGBoost Regressor RMSE: {rmse_xgb:.4f}")

XGBoost Regressor RMSE: 4.6289


## 마. 스태킹 (Stacking / Stacked Generalization)
### 1. **스태킹의 기본 원리**
* 학습용 데이터에 대한 개별 모델들의 예측 결과들을 새로운 특성(Feature)으로 만들어 메타 모델(최종 예측 모델)을 학습시키는 앙상블 기법

### 2. **스태킹 학습 2단계 구조**
* **1단계 (Base Models)**: 서로 다른 알고리즘(Logistic Regression, Decision Tree 등)으로 개별 모델 학습 및 예측 수행
* **2단계 (Meta Model)**: 1단계 모델들의 예측값들을 입력 데이터(X)로 삼아 최종 메타 모델(예: Logistic Regression / LinearRegression)을 훈련
* **정리**: `[기반 모델들의 train]` $\rightarrow$ `[출력값 + 정답 결합]` $\rightarrow$ `[메타 모델의 train]`

### 3. **스태킹 예측 2단계 구조**
* **1단계 (Base Models)**: 서로 다른 알고리즘으로 평가용 데이터에 대한 예측 수행
* **2단계 (Meta Model)**: 1단계 모델들의 예측값들을 입력 데이터(X)로 삼아 최종 메타 모델 예측
* **정리**: `[기반 모델들의 predict]` $\rightarrow$ `[출력값 결합]` $\rightarrow$ `[메타 모델의 predict]`

In [23]:
# 스태킹 분류

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

In [25]:
# 2. 1단계: 개별 기본 모델(Base Estimators) 설정
base_estimators = [
    ('rf', RandomForestClassifier(random_state=0)),
    ('xgb', XGBClassifier(random_state=0)),
    ('dt', DecisionTreeClassifier(random_state=0))
]

# 3. 2단계: 최종 메타 모델(Meta Learner / Blender) 설정
final_meta_learner = LogisticRegression(max_iter=1000, random_state=0)

In [26]:
# 4. Stacking 분류 모델 생성 및 학습
stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=final_meta_learner
)

stack_clf.fit(X_train_c, y_train_c)
print()

In [27]:
# 5. 평가용 데이터를 이용한 예측 및 성능 평가 (Accuracy)
pred_test_stack = stack_clf.predict(X_test_c)
acc_stack = accuracy_score(y_test_c, pred_test_stack)

print(f"Stacking Classifier Accuracy: {acc_stack:.4f} ({acc_stack * 100:.2f}%)")

Stacking Classifier Accuracy: 0.7482 (74.82%)


In [28]:
# 스태킹 회귀

In [29]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

In [30]:
# 2. 1단계: 개별 기본 회귀 모델(Base Estimators) 설정
base_regressors = [
    ('rf', RandomForestRegressor(random_state=0)),
    ('xgb', XGBRegressor(random_state=0)),
    ('dt', DecisionTreeRegressor(random_state=0))
]

# 3. 2단계: 최종 메타 모델(Meta Learner / Blender) 설정 (회귀이므로 선형 회귀 사용)
final_meta_regressor = LinearRegression()

In [31]:
# 4. Stacking 회귀 모델 생성 및 학습
stack_reg = StackingRegressor(
    estimators=base_regressors,
    final_estimator=final_meta_regressor
)

stack_reg.fit(X_train_r, y_train_r)
print()

In [32]:
# 5. 평가용 데이터를 이용한 예측 및 성능 평가 (RMSE)
pred_test_stack_reg = stack_reg.predict(X_test_r)
rmse_stack = rmse(y_test_r, pred_test_stack_reg)

print(f"Stacking Regressor RMSE: {rmse_stack:.4f}")

Stacking Regressor RMSE: 4.4784
